In [1]:
pip install rdkit pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 32.3 MB/s eta 0:00:00


In [2]:
import pandas as pd

# Load your existing processed dataset
df = pd.read_csv("Supplementary_4.csv")

# Extract unique SMILES and save as training_smiles.csv
training_smiles = df[['canonical_smiles']].drop_duplicates().dropna()
training_smiles.to_csv("training_smiles.csv", index=False)

print(f"Saved {len(training_smiles)} unique training SMILES")
# Expected: ~40,000–50,000 unique structures

Saved 41028 unique training SMILES


In [3]:
"""
Virtual Screening Analysis Pipeline
=====================================
Filters broad-spectrum antibacterial hits from COCONUT/NPAtlas screening results.

Pipeline:
  1. Load combined screening results (score >= 0.80)
  2. Calculate physicochemical properties
  3. Apply Egan AND Veber druglikeness filters (MW <= 500)
  4. Deduplicate by InChIKey
  5. Tanimoto similarity filter against training set (removes near-copies of known drugs)
  6. Tier results into 3 confidence levels
  7. Save tiered outputs + full supplementary file
  8. Print all 19 novel candidates to screen

Requirements:
    pip install rdkit pandas numpy
"""

import pandas as pd
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from rdkit.Chem.inchi import MolToInchiKey
import os
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIGURATION — edit paths here
# ─────────────────────────────────────────────

HITS_CSV         = "Supplementary_5.csv"
TRAINING_CSV     = "Supplementary_4.csv"
OUTPUT_DIR       = "results"

# Thresholds
SCORE_THRESHOLD    = 0.80   # minimum combined antibacterial score
TANIMOTO_THRESHOLD = 0.40   # above this = known-like, below = novel
MW_CUTOFF          = 500    # maximum molecular weight

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ─────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────

def mol_from_smiles(smiles):
    """Safely parse SMILES, return None if invalid."""
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    return Chem.MolFromSmiles(smiles)


def calculate_properties(smiles):
    """
    Calculate physicochemical descriptors.
    Returns dict or None if SMILES is unparseable.
    """
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return {
        "MW":             round(Descriptors.MolWt(mol), 2),
        "LogP":           round(Descriptors.MolLogP(mol), 2),
        "TPSA":           round(rdMolDescriptors.CalcTPSA(mol), 2),
        "RotatableBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "HBD":            rdMolDescriptors.CalcNumHBD(mol),
        "HBA":            rdMolDescriptors.CalcNumHBA(mol),
        "InChIKey":       MolToInchiKey(mol),
    }


def apply_druglikeness(row):
    """
    Egan rule:  LogP <= 5.88 AND TPSA <= 131.6
    Veber rule: RotatableBonds <= 10 AND TPSA <= 140
    Both must pass AND MW <= 500.
    """
    egan  = (row["LogP"] <= 5.88) and (row["TPSA"] <= 131.6)
    veber = (row["RotatableBonds"] <= 10) and (row["TPSA"] <= 140)
    mw_ok = row["MW"] <= MW_CUTOFF
    return egan, veber, (egan and veber and mw_ok)


def get_morgan_fp(smiles, radius=2, nbits=2048):
    """Return Morgan fingerprint or None."""
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)


def max_tanimoto_to_set(query_fp, reference_fps):
    """Return highest Tanimoto similarity between query and any reference fp."""
    if query_fp is None or not reference_fps:
        return None
    sims = DataStructs.BulkTanimotoSimilarity(query_fp, reference_fps)
    return round(max(sims), 4)


def assign_tier(score):
    if score >= 0.95:
        return "Tier 1 (>=0.95)"
    elif score >= 0.90:
        return "Tier 2 (0.90-0.95)"
    else:
        return "Tier 3 (0.80-0.90)"


# ─────────────────────────────────────────────
# STEP 1 — Load hits
# ─────────────────────────────────────────────

print("=" * 60)
print("VIRTUAL SCREENING ANALYSIS PIPELINE")
print("=" * 60)

df = pd.read_csv(HITS_CSV)
print(f"\n[1] Loaded {len(df):,} compounds from {HITS_CSV}")

df = df[df["combined_antibacterial_score"] >= SCORE_THRESHOLD].copy()
print(f"    After score >= {SCORE_THRESHOLD} filter: {len(df):,} compounds")


# ─────────────────────────────────────────────
# STEP 2 — Calculate properties + druglikeness
# ─────────────────────────────────────────────

print("\n[2] Calculating physicochemical properties...")

records = []
skipped = 0

for _, row in df.iterrows():
    props = calculate_properties(row["canonical_smiles"])
    if props is None:
        skipped += 1
        continue

    egan, veber, passes = apply_druglikeness({
        "LogP":           props["LogP"],
        "TPSA":           props["TPSA"],
        "RotatableBonds": props["RotatableBonds"],
        "MW":             props["MW"]
    })

    records.append({
        "compound_id":                  row["compound_id"],
        "compound_name":                row.get("compound_name", ""),
        "canonical_smiles":             row["canonical_smiles"],
        "InChIKey":                     props["InChIKey"],
        "MW":                           props["MW"],
        "LogP":                         props["LogP"],
        "TPSA":                         props["TPSA"],
        "RotatableBonds":               props["RotatableBonds"],
        "HBD":                          props["HBD"],
        "HBA":                          props["HBA"],
        "Egan_Pass":                    egan,
        "Veber_Pass":                   veber,
        "Passes_Druglikeness":          passes,
        "ecoli_probability":            row.get("ecoli_probability"),
        "paeruginosa_probability":      row.get("paeruginosa_probability"),
        "saureus_probability":          row.get("saureus_probability"),
        "combined_antibacterial_score": row["combined_antibacterial_score"],
    })

result_df = pd.DataFrame(records)
print(f"    Parsed: {len(result_df):,} | Skipped (bad SMILES): {skipped}")
print(f"    Egan pass:  {result_df['Egan_Pass'].sum():,}")
print(f"    Veber pass: {result_df['Veber_Pass'].sum():,}")
print(f"    Both pass (Egan AND Veber, MW<={MW_CUTOFF}): {result_df['Passes_Druglikeness'].sum():,}")


# ─────────────────────────────────────────────
# STEP 3 — Deduplicate by InChIKey
# ─────────────────────────────────────────────

print("\n[3] Deduplicating by InChIKey...")

before = len(result_df)
result_df = (result_df
             .sort_values("combined_antibacterial_score", ascending=False)
             .drop_duplicates(subset="InChIKey", keep="first")
             .reset_index(drop=True))

print(f"    Removed {before - len(result_df):,} duplicate structures")
print(f"    Unique structures: {len(result_df):,}")

druglike_df = result_df[result_df["Passes_Druglikeness"]].copy()
print(f"    Unique druglike structures: {len(druglike_df):,}")


# ─────────────────────────────────────────────
# STEP 4 — Tanimoto similarity filter
# ─────────────────────────────────────────────

print("\n[4] Running Tanimoto similarity filter against training set...")

training_fps = []
if os.path.exists(TRAINING_CSV):
    train_df = pd.read_csv(TRAINING_CSV)
    print(f"    Training set loaded: {len(train_df):,} compounds")
    for smi in train_df["canonical_smiles"].dropna():
        fp = get_morgan_fp(smi)
        if fp is not None:
            training_fps.append(fp)
    print(f"    Training fingerprints generated: {len(training_fps):,}")
else:
    print(f"    WARNING: Training set not found at {TRAINING_CSV}")
    print(f"    Skipping Tanimoto filter — all compounds marked as 'similarity unknown'")

print("    Calculating similarities (this may take a few minutes)...")
druglike_df["Max_Tanimoto_to_Training"] = druglike_df["canonical_smiles"].apply(
    lambda smi: max_tanimoto_to_set(get_morgan_fp(smi), training_fps) if training_fps else None
)

if training_fps:
    druglike_df["Novelty_Flag"] = druglike_df["Max_Tanimoto_to_Training"].apply(
        lambda t: "Known-like" if (t is not None and t > TANIMOTO_THRESHOLD)
                  else "Novel candidate"
    )
    novel_count = (druglike_df["Novelty_Flag"] == "Novel candidate").sum()
    known_count = (druglike_df["Novelty_Flag"] == "Known-like").sum()
    print(f"    Novel candidates (Tanimoto <= {TANIMOTO_THRESHOLD}): {novel_count:,}")
    print(f"    Known-like (Tanimoto > {TANIMOTO_THRESHOLD}):        {known_count:,}")
else:
    druglike_df["Novelty_Flag"] = "Similarity unknown (no training set)"
    novel_count = 0
    known_count = 0


# ─────────────────────────────────────────────
# STEP 5 — Tier assignment
# ─────────────────────────────────────────────

print("\n[5] Assigning confidence tiers...")

druglike_df["Tier"] = druglike_df["combined_antibacterial_score"].apply(assign_tier)
print(druglike_df["Tier"].value_counts().to_string())


# ─────────────────────────────────────────────
# STEP 6 — Save all outputs
# ─────────────────────────────────────────────

print("\n[6] Saving outputs...")

# S1 — Full supplementary file (all tiers, all druglike)
supp_path = os.path.join(OUTPUT_DIR, "S1_all_druglike_hits.csv")
druglike_df.sort_values("combined_antibacterial_score", ascending=False).to_csv(supp_path, index=False)
print(f"    Supplementary (all tiers):   {supp_path}  ({len(druglike_df):,} compounds)")

# Tier 1 — model validation / positive controls
tier1 = druglike_df[druglike_df["Tier"] == "Tier 1 (>=0.95)"]
t1_path = os.path.join(OUTPUT_DIR, "tier1_score_gte095.csv")
tier1.sort_values("combined_antibacterial_score", ascending=False).to_csv(t1_path, index=False)
print(f"    Tier 1 (>=0.95):             {t1_path}  ({len(tier1):,} compounds)")

# Tier 2 — high confidence candidates
tier2 = druglike_df[druglike_df["Tier"] == "Tier 2 (0.90-0.95)"]
t2_path = os.path.join(OUTPUT_DIR, "tier2_score_090_095.csv")
tier2.sort_values("combined_antibacterial_score", ascending=False).to_csv(t2_path, index=False)
print(f"    Tier 2 (0.90-0.95):          {t2_path}  ({len(tier2):,} compounds)")

# Tier 3 — broader candidates
tier3 = druglike_df[druglike_df["Tier"] == "Tier 3 (0.80-0.90)"]
t3_path = os.path.join(OUTPUT_DIR, "tier3_score_080_090.csv")
tier3.sort_values("combined_antibacterial_score", ascending=False).to_csv(t3_path, index=False)
print(f"    Tier 3 (0.80-0.90):          {t3_path}  ({len(tier3):,} compounds)")

# Novel priority candidates — ALL tiers, novel flag only (FIXED)
if training_fps:
    novel_hits = druglike_df[
        druglike_df["Novelty_Flag"] == "Novel candidate"
    ].sort_values("combined_antibacterial_score", ascending=False)

    novel_path = os.path.join(OUTPUT_DIR, "novel_priority_candidates.csv")
    novel_hits.to_csv(novel_path, index=False)
    print(f"    Novel priority candidates:   {novel_path}  ({len(novel_hits):,} compounds)")


# ─────────────────────────────────────────────
# STEP 7 — Print novel candidates to screen
# ─────────────────────────────────────────────

if training_fps and len(novel_hits) > 0:
    print("\n" + "=" * 60)
    print(f"NOVEL PRIORITY CANDIDATES ({len(novel_hits)} compounds)")
    print("=" * 60)

    display_cols = [
        "compound_id", "compound_name",
        "ecoli_probability", "paeruginosa_probability", "saureus_probability",
        "combined_antibacterial_score",
        "MW", "LogP", "TPSA",
        "Max_Tanimoto_to_Training", "Tier"
    ]

    pd.set_option("display.max_rows", 50)
    pd.set_option("display.max_columns", 20)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 40)

    print(novel_hits[display_cols].to_string(index=False))

    print("\n--- SMILES for each novel candidate ---")
    for _, r in novel_hits.iterrows():
        name = r["compound_name"] if pd.notna(r["compound_name"]) and r["compound_name"] != "" else "Unnamed"
        print(f"\n{r['compound_id']} | {name}")
        print(f"  SMILES: {r['canonical_smiles']}")
        print(f"  Score:  {r['combined_antibacterial_score']:.4f} | "
              f"E.coli: {r['ecoli_probability']:.3f} | "
              f"P.aer: {r['paeruginosa_probability']:.3f} | "
              f"S.aur: {r['saureus_probability']:.3f}")
        print(f"  MW: {r['MW']} | LogP: {r['LogP']} | TPSA: {r['TPSA']} | "
              f"Tanimoto: {r['Max_Tanimoto_to_Training']}")

elif training_fps and len(novel_hits) == 0:
    print("\nNo novel candidates found. Consider lowering TANIMOTO_THRESHOLD (e.g. to 0.50).")
else:
    print("\nTanimoto filter was not run (no training set). Cannot identify novel candidates.")


# ─────────────────────────────────────────────
# STEP 8 — Final summary
# ─────────────────────────────────────────────

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"  Total input hits (score >= {SCORE_THRESHOLD}):    {len(df):,}")
print(f"  Bad SMILES removed:                {skipped:,}")
print(f"  After InChIKey deduplication:      {len(result_df):,}")
print(f"  Passing Egan AND Veber (MW<={MW_CUTOFF}):  {len(druglike_df):,}")
if training_fps:
    print(f"  Novel (Tanimoto <= {TANIMOTO_THRESHOLD}):           {novel_count:,}")
    print(f"  Known-like (Tanimoto > {TANIMOTO_THRESHOLD}):       {known_count:,}")
print()
print("  Score tiers (druglike, unique):")
for tier, count in sorted(druglike_df["Tier"].value_counts().items()):
    print(f"    {tier}: {count:,}")
print()
print(f"  Output directory: {OUTPUT_DIR}")
print("=" * 60)
print("Done.")

VIRTUAL SCREENING ANALYSIS PIPELINE

[1] Loaded 1,791 compounds from Supplementary_5.csv
    After score >= 0.8 filter: 1,791 compounds

[2] Calculating physicochemical properties...
    Parsed: 1,791 | Skipped (bad SMILES): 0
    Egan pass:  470
    Veber pass: 476
    Both pass (Egan AND Veber, MW<=500): 402

[3] Deduplicating by InChIKey...
    Removed 260 duplicate structures
    Unique structures: 1,531
    Unique druglike structures: 365

[4] Running Tanimoto similarity filter against training set...
    Training set loaded: 41,028 compounds


Streaming output truncated to the last 5000 lines.
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:20:42] DEPRECATION WARNING: please use MorganGenerator
[13:2

    Training fingerprints generated: 41,027
    Calculating similarities (this may take a few minutes)...


[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerator
[13:20:44] DEPRECATION WARNING: please use MorganGenerat

    Novel candidates (Tanimoto <= 0.4): 19
    Known-like (Tanimoto > 0.4):        346

[5] Assigning confidence tiers...
Tier
Tier 3 (0.80-0.90)    242
Tier 1 (>=0.95)        86
Tier 2 (0.90-0.95)     37

[6] Saving outputs...
    Supplementary (all tiers):   results/S1_all_druglike_hits.csv  (365 compounds)
    Tier 1 (>=0.95):             results/tier1_score_gte095.csv  (86 compounds)
    Tier 2 (0.90-0.95):          results/tier2_score_090_095.csv  (37 compounds)
    Tier 3 (0.80-0.90):          results/tier3_score_080_090.csv  (242 compounds)
    Novel priority candidates:   results/novel_priority_candidates.csv  (19 compounds)

NOVEL PRIORITY CANDIDATES (19 compounds)
 compound_id   compound_name  ecoli_probability  paeruginosa_probability  saureus_probability  combined_antibacterial_score     MW  LogP   TPSA  Max_Tanimoto_to_Training               Tier
CNP0498636.1 NCGC00393411-01              0.844                 0.804000             0.936623                      0.861541 384.

[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator
[13:20:50] DEPRECATION WARNING: please use MorganGenerator


In [4]:
known_like_hits = druglike_df[
    druglike_df["Novelty_Flag"] == "Known-like"
].sort_values("combined_antibacterial_score", ascending=False)

known_like_path = os.path.join(OUTPUT_DIR, "known_like_compounds.csv")
known_like_hits.to_csv(known_like_path, index=False)
print(f"    Known-like compounds:        {known_like_path}  ({len(known_like_hits):,} compounds)")

    Known-like compounds:        results/known_like_compounds.csv  (346 compounds)


In [ ]:
"""
PubChem Automated Annotation Pipeline
=======================================
For each novel candidate, this script:
  1. Searches PubChem by SMILES (exact match) to get CID
  2. Fetches compound name, synonyms, molecular formula
  3. Fetches bioactivity assay count (any recorded biological activity)
  4. Fetches any antibacterial / antimicrobial annotations
  5. Classifies each compound into:
       - Category A: Known antibacterial activity reported
       - Category B: Other biological activity but no antibacterial data
       - Category C: No biological activity recorded (priority novel hit)
  6. Saves annotated table ready for paper

Requirements:
    pip install pandas requests rdkit

Note: Uses PubChem PUG REST API (free, no key needed).
      Adds 0.5s delay between requests to respect rate limits.
"""

import pandas as pd
import requests
import time
import json
from rdkit import Chem
from rdkit.Chem.inchi import MolToInchi, MolToInchiKey
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

INPUT_CSV  = "novel_priority_candidates.csv"
OUTPUT_CSV = "novel_candidates_annotated.csv"
OUTPUT_TSV = "novel_candidates_paper_table.tsv"

DELAY = 0.5   # seconds between API calls (respect PubChem rate limit)

ANTIBACTERIAL_KEYWORDS = [
    "antibacterial", "antimicrobial", "antibiotic",
    "bactericidal", "bacteriostatic",
    "staphylococcus", "escherichia", "pseudomonas",
    "gram-positive", "gram-negative",
    "MIC", "minimum inhibitory"
]


# ─────────────────────────────────────────────
# PUBCHEM API FUNCTIONS
# ─────────────────────────────────────────────

def smiles_to_inchikey(smiles):
    """Convert SMILES to InChIKey using RDKit."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MolToInchiKey(mol)


def get_cid_from_inchikey(inchikey):
    """Search PubChem by InChIKey, return CID or None."""
    if not inchikey:
        return None
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/cids/JSON"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            data = r.json()
            cids = data.get("IdentifierList", {}).get("CID", [])
            return cids[0] if cids else None
        return None
    except Exception:
        return None


def get_cid_from_smiles(smiles):
    """Search PubChem by SMILES, return CID or None."""
    url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/cids/JSON"
    try:
        r = requests.post(url, data={"smiles": smiles}, timeout=15)
        if r.status_code == 200:
            data = r.json()
            cids = data.get("IdentifierList", {}).get("CID", [])
            return cids[0] if cids else None
        return None
    except Exception:
        return None


def get_compound_info(cid):
    """Fetch compound name, synonyms, and formula from PubChem CID."""
    if not cid:
        return {
            "pubchem_cid": None,
            "pubchem_name": "Not found in PubChem",
            "molecular_formula": None,
            "synonyms": [],
            "iupac_name": None
        }

    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/JSON"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            return {"pubchem_cid": cid, "pubchem_name": None,
                    "molecular_formula": None, "synonyms": [], "iupac_name": None}

        data = r.json()
        props = data.get("PC_Compounds", [{}])[0].get("props", [])

        name, formula, iupac = None, None, None
        for p in props:
            urn = p.get("urn", {})
            val = p.get("value", {})
            label = urn.get("label", "")
            name_field = urn.get("name", "")

            if label == "IUPAC Name" and name_field == "Preferred":
                iupac = val.get("sval")
            if label == "Molecular Formula":
                formula = val.get("sval")

        return {
            "pubchem_cid": cid,
            "pubchem_name": iupac,
            "molecular_formula": formula,
            "synonyms": [],
            "iupac_name": iupac
        }
    except Exception:
        return {"pubchem_cid": cid, "pubchem_name": None,
                "molecular_formula": None, "synonyms": [], "iupac_name": None}


def get_synonyms(cid):
    """Fetch synonyms for a PubChem CID."""
    if not cid:
        return []
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            data = r.json()
            syns = data.get("InformationList", {}).get("Information", [{}])[0].get("Synonym", [])
            return syns[:10]  # return first 10 synonyms
        return []
    except Exception:
        return []


def get_bioassay_count(cid):
    """Get total number of bioassay records for a CID."""
    if not cid:
        return 0
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/assaysummary/JSON"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            data = r.json()
            table = data.get("Table", {})
            rows = table.get("Row", [])
            return len(rows)
        return 0
    except Exception:
        return 0


def get_active_assay_count(cid):
    """Get number of assays where compound was ACTIVE."""
    if not cid:
        return 0
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/assaysummary/JSON"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            data = r.json()
            rows = data.get("Table", {}).get("Row", [])
            # Each row is a dict; outcome is in columns
            # PubChem returns columns list separately
            cols = data.get("Table", {}).get("Columns", {}).get("Column", [])
            if "Bioactivity Outcome" in cols:
                outcome_idx = cols.index("Bioactivity Outcome")
                active = sum(
                    1 for row in rows
                    if row.get("Cell", [])[outcome_idx] == "Active"
                    if len(row.get("Cell", [])) > outcome_idx
                )
                return active
        return 0
    except Exception:
        return 0


def check_antibacterial_activity(cid, synonyms):
    """
    Check if compound has any antibacterial activity reported.
    Searches synonym list and PubChem bioassay descriptions.
    Returns: (has_antibacterial, evidence_string)
    """
    # First check synonyms for antibiotic keywords
    syn_text = " ".join(synonyms).lower()
    for kw in ANTIBACTERIAL_KEYWORDS:
        if kw.lower() in syn_text:
            return True, f"Keyword '{kw}' found in synonyms"

    if not cid:
        return False, "No PubChem CID found"

    # Check bioassay descriptions
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/assaysummary/JSON"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            text = r.text.lower()
            for kw in ANTIBACTERIAL_KEYWORDS:
                if kw.lower() in text:
                    return True, f"Keyword '{kw}' found in bioassay records"
        return False, "No antibacterial activity found in PubChem"
    except Exception:
        return False, "API error during bioassay check"


def classify_compound(cid, has_antibacterial, total_assays, synonyms):
    """
    Category A: Known antibacterial activity reported
    Category B: Other biological activity, no antibacterial data
    Category C: No biological activity — priority novel hit
    """
    if has_antibacterial:
        return "A", "Known antibacterial activity in literature"
    elif total_assays > 0:
        return "B", f"Biological activity reported ({total_assays} assays) but no antibacterial data"
    elif cid is None:
        return "C", "Not found in PubChem — uncharacterized compound"
    else:
        return "C", f"Found in PubChem (CID:{cid}) but no biological activity recorded"


# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────

print("=" * 65)
print("PUBCHEM ANNOTATION PIPELINE FOR NOVEL CANDIDATES")
print("=" * 65)

df = pd.read_csv(INPUT_CSV)
print(f"\nLoaded {len(df)} novel candidates\n")

results = []

for i, row in df.iterrows():
    cid_label    = row["compound_id"]
    name         = row.get("compound_name", "")
    smiles       = row["canonical_smiles"]
    inchikey     = row.get("InChIKey", smiles_to_inchikey(smiles))

    print(f"[{i+1:02d}/{len(df)}] {cid_label} | {name if pd.notna(name) and name != '' else 'Unnamed'}")

    # Step 1 — Get PubChem CID (try InChIKey first, then SMILES)
    cid = get_cid_from_inchikey(inchikey)
    time.sleep(DELAY)

    if not cid:
        cid = get_cid_from_smiles(smiles)
        time.sleep(DELAY)

    print(f"         PubChem CID: {cid if cid else 'Not found'}")

    # Step 2 — Get compound info
    info = get_compound_info(cid)
    time.sleep(DELAY)

    # Step 3 — Get synonyms
    synonyms = get_synonyms(cid)
    time.sleep(DELAY)

    # Step 4 — Get bioassay count
    total_assays = get_bioassay_count(cid)
    time.sleep(DELAY)

    # Step 5 — Check antibacterial activity
    has_antibacterial, ab_evidence = check_antibacterial_activity(cid, synonyms)
    time.sleep(DELAY)

    # Step 6 — Classify
    category, category_note = classify_compound(cid, has_antibacterial, total_assays, synonyms)

    print(f"         Bioassays: {total_assays} | Antibacterial: {has_antibacterial} | Category: {category}")
    print(f"         Note: {category_note}")

    results.append({
        # Identity
        "compound_id":                  cid_label,
        "coconut_name":                 name if pd.notna(name) and name != "" else "Unnamed",
        "pubchem_cid":                  cid,
        "pubchem_iupac_name":           info.get("iupac_name"),
        "molecular_formula":            info.get("molecular_formula"),
        "synonyms_top5":                " | ".join(synonyms[:5]),
        "canonical_smiles":             smiles,
        "InChIKey":                     inchikey,

        # Physicochemical
        "MW":                           row["MW"],
        "LogP":                         row["LogP"],
        "TPSA":                         row["TPSA"],
        "HBD":                          row["HBD"],
        "HBA":                          row["HBA"],
        "RotatableBonds":               row["RotatableBonds"],

        # Prediction scores
        "ecoli_probability":            row["ecoli_probability"],
        "paeruginosa_probability":      row["paeruginosa_probability"],
        "saureus_probability":          row["saureus_probability"],
        "combined_antibacterial_score": row["combined_antibacterial_score"],
        "Max_Tanimoto_to_Training":     row["Max_Tanimoto_to_Training"],
        "Tier":                         row["Tier"],

        # Annotation
        "total_pubchem_assays":         total_assays,
        "antibacterial_reported":       has_antibacterial,
        "antibacterial_evidence":       ab_evidence,
        "category":                     category,
        "category_description":         category_note,
    })

    print()

# ─────────────────────────────────────────────
# SAVE RESULTS
# ─────────────────────────────────────────────

out_df = pd.DataFrame(results)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nFull annotated table saved: {OUTPUT_CSV}")

# Clean paper table (key columns only)
paper_cols = [
    "compound_id", "coconut_name", "pubchem_cid",
    "molecular_formula", "MW", "LogP", "TPSA",
    "ecoli_probability", "paeruginosa_probability", "saureus_probability",
    "combined_antibacterial_score", "Max_Tanimoto_to_Training",
    "total_pubchem_assays", "antibacterial_reported",
    "category", "category_description", "canonical_smiles"
]
paper_df = out_df[paper_cols]
paper_df.to_csv(OUTPUT_TSV, sep="\t", index=False)
print(f"Paper table saved:          {OUTPUT_TSV}")

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────

print("\n" + "=" * 65)
print("ANNOTATION SUMMARY")
print("=" * 65)

cat_counts = out_df["category"].value_counts()
print(f"\n  Category A (known antibacterial):      {cat_counts.get('A', 0)}")
print(f"  Category B (other bio activity):       {cat_counts.get('B', 0)}")
print(f"  Category C (uncharacterized novel):    {cat_counts.get('C', 0)}")

print("\n  --- Category A (positive controls) ---")
cat_a = out_df[out_df["category"] == "A"][["compound_id","coconut_name","combined_antibacterial_score","antibacterial_evidence"]]
if len(cat_a):
    print(cat_a.to_string(index=False))
else:
    print("  None")

print("\n  --- Category B (underexplored) ---")
cat_b = out_df[out_df["category"] == "B"][["compound_id","coconut_name","combined_antibacterial_score","total_pubchem_assays","category_description"]]
if len(cat_b):
    print(cat_b.to_string(index=False))
else:
    print("  None")

print("\n  --- Category C (priority novel hits) ---")
cat_c = out_df[out_df["category"] == "C"][["compound_id","coconut_name","pubchem_cid","combined_antibacterial_score","category_description"]]
if len(cat_c):
    print(cat_c.to_string(index=False))
else:
    print("  None")

print("\n" + "=" * 65)
print("Done. Share the output above to proceed with paper writing.")
print("=" * 65)

PUBCHEM ANNOTATION PIPELINE FOR NOVEL CANDIDATES

Loaded 19 novel candidates

[01/19] CNP0498636.1 | NCGC00393411-01
         PubChem CID: 75367585
         Bioassays: 15 | Antibacterial: True | Category: A
         Note: Known antibacterial activity in literature

[02/19] CNP0332203.1 | Unnamed
         PubChem CID: 75367535
         Bioassays: 0 | Antibacterial: False | Category: C
         Note: Found in PubChem (CID:75367535) but no biological activity recorded

[03/19] CNP0551375.1 | NCGC00393419-01
         PubChem CID: 75367534
         Bioassays: 15 | Antibacterial: True | Category: A
         Note: Known antibacterial activity in literature

[04/19] CNP0573332.0 | Unnamed
         PubChem CID: 137918915
         Bioassays: 0 | Antibacterial: False | Category: C
         Note: Found in PubChem (CID:137918915) but no biological activity recorded

[05/19] CNP0547734.0 | Unnamed
         PubChem CID: 137918917
         Bioassays: 0 | Antibacterial: False | Category: C
         Not